# Image-stream variance

Reads an image array `(N_frames, H, W)` from the input run, computes
per-pixel variance along the time axis, and writes one image of shape
`(H, W)` back to Tiled as a new run that inherits the input's
access tags plus the plugin's output_tags.

In [ ]:
# Defaults are overridden by papermill at execution time.
stream = 'primary'
field = 'image'
dtype = 'float32'

In [ ]:
import json
import os
import uuid

import numpy as np
import scrapbook as sb
from tiled.client import from_uri

from lucid_pipelines.notebook import (
    get_input_access_blob,
    get_input_run,
    get_provenance,
)

In [ ]:
# Locate the image array on the input run. We try the common bluesky V3
# layout first (`<stream>.read()` returns a table; pull `field` out of
# it) and fall back to a direct child lookup so non-bluesky-shaped runs
# (e.g. a previously-written derived run) still work.

input_run = get_input_run()
stream_node = input_run[stream]

try:
    table = stream_node.read()
    column = table[field]
    frames = np.stack([np.asarray(x) for x in column.values])
except (TypeError, KeyError, AttributeError):
    # Stream isn't a table; try a direct node read.
    frames = np.asarray(stream_node[field][:])

if frames.ndim < 3:
    raise ValueError(
        f'expected >=3D image stack, got shape {frames.shape}'
    )

variance = frames.var(axis=0).astype(np.dtype(dtype))
print(f'variance shape={variance.shape} dtype={variance.dtype}')

In [ ]:
# Merge access tags: input run tags + plugin output_tags. Order-preserving
# dedup so the resulting list mirrors the human reading order.

input_blob = get_input_access_blob()
output_tags = json.loads(os.environ.get('LUCID_OUTPUT_TAGS', '[]'))
input_uid = os.environ['LUCID_INPUT_RUN_UID']

merged_tags = list(dict.fromkeys(
    list(input_blob.get('tags') or []) + list(output_tags)
))

In [ ]:
# Write the derived run. We use Tiled's create_container directly rather
# than bluesky's TiledWriter because the output is a single image with
# no descriptors / events — going through a synthetic RunEngine for one
# array is more ceremony than the PoC needs. `access_tags` is the load-
# bearing parameter for ALSAccessPolicy: it lands in `nodes.access_blob`
# which is what `AccessBlobFilter` reads.

client = from_uri(
    os.environ['TILED_URL'], api_key=os.environ['TILED_API_KEY'],
)

output_uid = str(uuid.uuid4())
run = client.create_container(
    key=output_uid,
    metadata={
        'start': {
            'uid': output_uid,
            'parent_run_uid': input_uid,
            'pipeline_provenance': get_provenance(),
            'tiled_access_tags': merged_tags,
        },
    },
    access_tags=merged_tags,
)
run.write_array(variance, key='variance')
print(f'wrote run {output_uid} with tags={merged_tags}')

In [ ]:
# Scrapbook glue is how the executor harvests output uids from the
# executed notebook.
sb.glue('output_run_uids', [output_uid])
sb.glue('metrics', {
    'n_frames': int(frames.shape[0]),
    'image_shape': list(variance.shape),
    'mean_variance': float(variance.mean()),
})